In [2]:
from langchain_community.document_loaders import JSONLoader
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.documents import Document
import json
import os

c:\Users\alam\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
if os.path.exists(r'C:\Users\alam\OneDrive - New York State Thruway Authority\Documents\Python\allrecipes\allrecipe_data.json'):
    with open(r'C:\Users\alam\OneDrive - New York State Thruway Authority\Documents\Python\allrecipes\allrecipe_data.json') as file:
        existing_data = json.load(file)

In [4]:
for item in existing_data:
    print(item)

Dinner Recipes
Fruits, Vegetables and Other Produce
Bread Recipes
Everyday Cooking
Lunch Recipes
U.S. Recipes
Appetizers and Snacks
Drinks
Breakfast and Brunch
Desserts
Main Dishes
Side Dishes
Healthy Recipes
Holidays and Events
Cuisines
BBQ & Grilling
Meat and Poultry
Seafood Recipes
Soups, Stews and Chili
Pasta and Noodles
Salad Recipes


In [ ]:
import os
os.chdir("../")

In [ ]:
%pwd

In [ ]:
file = './research/allrecipe_data.json'

In [ ]:
from langchain.schema import Document
if os.path.exists(file):
    with open(file) as file:
        data = json.load(file)


In [ ]:
#convert json file into Documents
loader = JSONLoader(
    file_path='./research/allrecipe_data.json',
    jq_schema=".",
    json_lines=False,
    text_content=False

)
data = loader.load()


In [ ]:
#From document extracted only source and page content
from typing import List
from langchain.schema import Document
def extracted_require_data(docs: List[Document]) -> List[Document]:
    minimal_docs: List[Document]=[]
    for doc in docs:
        src = doc.metadata.get("source")
        minimal_docs.append(
            Document(
                page_content = doc.page_content,
                metadata ={"source": src}
            )
        )
    return minimal_docs

In [ ]:
minimal_docs = extracted_require_data(data)

In [ ]:
from langchain.document_loaders import PyPDFLoader, DirectoryLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
#split the Data into Text chunk
text_splitter = RecursiveCharacterTextSplitter(chunk_size = 1000, chunk_overlap = 20)
text_chunks = text_splitter.split_documents(minimal_docs)


In [ ]:
#Embedding The text
from langchain.embeddings import HuggingFaceBgeEmbeddings
def Hugging_face_embedding():
    embedding = HuggingFaceBgeEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2')
    return embedding
embeddings = Hugging_face_embedding()

In [ ]:
from dotenv import load_dotenv
load_dotenv()

In [ ]:
PINECONE_API_KEY=os.environ.get('PINECONE_API_KEY')
OPENAI_API_KEY =os.environ.get('OPENAI_API_KEY ')

In [ ]:
#initialize the pinecone
from pinecone import Pinecone
pinecone_api_key = PINECONE_API_KEY

pc = Pinecone(api_key = pinecone_api_key)

In [ ]:
#create index in pinecone
from pinecone import ServerlessSpec
index_name ="all-food-recipes"

if not pc.has_index(index_name):
    pc.create_index (
        name = index_name,
        dimension = 384,
        metric = "cosine",
        spec = ServerlessSpec(cloud="aws", region="us-east-1")
      )
index = pc.Index(index_name)

In [ ]:
#store data inside pincecone database
from langchain_pinecone import PineconeVectorStore
docsearch = PineconeVectorStore.from_documents(
    documents = text_chunks,
    index_name = index_name,
    embedding=embeddings
)

In [ ]:
#Load existing Index
from langchain_pinecone import PineconeVectorStore
docsearch = PineconeVectorStore.from_existing_index(
    index_name = index_name,
    embedding=embeddings
)

In [ ]:
retriever = docsearch.as_retriever(search_type = "similarity", search_kwargs={"k":10})

In [ ]:
retriever_docs = retriever.invoke("Give me some Dinner Recipes")
retriever_docs

In [ ]:
from langchain_openai import ChatOpenAI
chatModel = ChatOpenAI(model="gpt-4o")

In [ ]:
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

In [ ]:
system_prompt = (
    "You are an intelligent assistant specialized in answering questions about cooking tips and food recipes. "
    "Use the following pieces of retrieved context from pinecone to answer the user's question "
    "When showing food items, please list them in numbered order like a recipe menu. For example:"
    "Chili Sauce Pasta"
     "Fried Chicken"
     "Garlic Bread"
     "This helps users follow the suggestions step-by-step."
    "the question. If you don't know the answer, say that you "
    "don't know. Use three sentences maximum and keep the "
    "answer concise."
    "\n\n"
    "{context}"
)


prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
        ("human", "{input}"),
    ]
)

In [ ]:
question_answer_chain = create_stuff_documents_chain(chatModel, prompt)
rag_chain = create_retrieval_chain(retriever,question_answer_chain)

In [ ]:
response = rag_chain.invoke({"input":"can you show me some noodles recipes"})
print(response["answer"])